# 3. Document Classification Based on BOW

## 1. Preparing the 20 Newsgroups Data and Feature Extraction

 http://scikit-learn.org/0.19/datasets/twenty_newsgroups.html

In [ ]:
from sklearn.datasets import fetch_20newsgroups

# Create a list of topics to select from the 20 categories
categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space']

# Fetch the training dataset
newsgroups_train = fetch_20newsgroups(subset='train',
# Remove hinting parts from the email content - classify purely based on content
                                      remove=('headers', 'footers', 'quotes'),
                                      categories=categories)

# Fetch the test dataset
newsgroups_test = fetch_20newsgroups(subset='test',
                                     remove=('headers', 'footers', 'quotes'),
                                     categories=categories)

print('#Train set size:', len(newsgroups_train.data))
print('#Test set size:', len(newsgroups_test.data))
print('#Selected categories:', newsgroups_train.target_names)
print('#Train labels:', set(newsgroups_train.target))

#Train set size: 2034
#Test set size: 1353
#Selected categories: ['alt.atheism', 'comp.graphics', 'sci.space', 'talk.religion.misc']
#Train labels: {np.int64(0), np.int64(1), np.int64(2), np.int64(3)}


In [ ]:
print('#Train set text samples:', newsgroups_train.data[0])
print('#Train set label smaples:', newsgroups_train.target[0])
print('#Test set text samples:', newsgroups_test.data[0])
print('#Test set label smaples:', newsgroups_test.target[0])

#Train set text samples: Hi,

I've noticed that if you only save a model (with all your mapping planes
positioned carefully) to a .3DS file that when you reload it after restarting
3DS, they are given a default position and orientation.  But if you save
to a .PRJ file their positions/orientation are preserved.  Does anyone
know why this information is not stored in the .3DS file?  Nothing is
explicitly said in the manual about saving texture rules in the .PRJ file. 
I'd like to be able to read the texture rule information, does anyone have 
the format for the .PRJ file?

Is the .CEL file format available from somewhere?

Rych
#Train set label smaples: 1
#Test set text samples: TRry the SKywatch project in  Arizona.
#Test set label smaples: 2


In [ ]:
X_train = newsgroups_train.data   # Training dataset documents
y_train = newsgroups_train.target # Training dataset labels

X_test = newsgroups_test.data     # Test dataset documents
y_test = newsgroups_test.target   # Test dataset labels

## 2. Document Representation Based on Distributed Representation

### 1) Word2Vec

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 74.3 MB/s eta 0:00:00


In [ ]:
# Import necessary libraries for Word2Vec and machine learning models
import gensim
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from collections import Counter
from nltk.tokenize import word_tokenize
import nltk

# Download required NLTK resources for tokenization and stopwords
nltk.download('punkt')  # For word tokenization
nltk.download('stopwords')  # For filtering out common stopwords
nltk.download('punkt_tab')

# Initialize a set of stopwords for English and a stemmer to reduce words to their root form
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()  # PorterStemmer is commonly used to reduce words to their stem form

# Data preprocessing function
def preprocess_data(data):
    processed_data = []
    for sentence in data:
        # Tokenize the sentence into words
        tokens = word_tokenize(sentence)
        # Convert to lowercase, remove stopwords and special characters, and apply stemming
        tokens = [stemmer.stem(re.sub(r'\W+', '', word.lower()))
                  for word in tokens
                  if word.lower() not in stop_words and re.sub(r'\W+', '', word)]
        processed_data.append(tokens)  # Add the cleaned tokens to the processed data
    return processed_data

# Step 1: Train the Word2Vec model
# Preprocess the training and testing data using the preprocess_data function
X_train_tokenized = preprocess_data(X_train)
X_test_tokenized = preprocess_data(X_test)

# Flatten the tokenized training data into a single list and calculate word frequency
all_words = [word for sentence in X_train_tokenized for word in sentence]
word_counts = Counter(all_words)  # Count frequency of each word in the training data

# Define a threshold to remove low-frequency words
min_count_threshold = 2  # Words with a frequency of 2 or lower will be removed
# Filter the tokenized training and testing data to keep only frequent words
X_train_tokenized = [[word for word in sentence if word_counts[word] > min_count_threshold] for sentence in X_train_tokenized]
X_test_tokenized = [[word for word in sentence if word_counts[word] > min_count_threshold] for sentence in X_test_tokenized]

# Train the Word2Vec model with the tokenized and filtered training data
# vector_size: Dimensionality of the word vectors
# window: Maximum distance between the current and predicted word
# sg: Use skip-gram (1) instead of CBOW (0)
w2v_model = Word2Vec(sentences=X_train_tokenized, vector_size=100, window=2, min_count=2, sg=1)

# Step 2: Function to generate Word2Vec vectors for each sentence
# Each sentence vector is computed as the average of its word vectors
def get_w2v_vectors(data, model, vector_size=100):
    vectors = []
    for sentence in data:
        # Initialize a zero vector for the sentence
        sentence_vec = np.zeros(vector_size)
        count = 0  # Track the number of words found in the Word2Vec model
        for word in sentence:
            # Check if the word exists in the Word2Vec model
            if word in model.wv.key_to_index:
                # Add the word vector to the sentence vector
                sentence_vec += model.wv[word]
                count += 1
        # If the sentence contains valid words, compute the average of the word vectors
        if count != 0:
            # The sentence vector is the average of the word vectors in the sentence
            sentence_vec /= count
        # Append the resulting sentence vector (average of word vectors) to the list
        vectors.append(sentence_vec)
    return np.array(vectors)

# Generate Word2Vec vectors for both the training and testing data
X_train_w2v = get_w2v_vectors(X_train_tokenized, w2v_model)
X_test_w2v = get_w2v_vectors(X_test_tokenized, w2v_model)

# Step 3: Train machine learning models (Logistic Regression and Random Forest)
# Initialize two different classifiers for comparison
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),  # Logistic Regression with maximum iterations set to 1000
    'Random Forest': RandomForestClassifier()  # Random Forest Classifier
}

# Dictionary to store model names and their performance metrics
results = {'Model': [], 'Train Accuracy': [], 'Test Accuracy': []}

# Train each model and evaluate accuracy on the training and testing data
for model_name, model in models.items():
    # Fit the model using the Word2Vec vectorized training data and corresponding labels
    model.fit(X_train_w2v, y_train)
    # Calculate accuracy on both the training and testing datasets
    train_acc = model.score(X_train_w2v, y_train)
    test_acc = model.score(X_test_w2v, y_test)

    # Store the results for each model
    results['Model'].append(model_name + " (Word2Vec)")  # Add the model name and the method used (Word2Vec)
    results['Train Accuracy'].append(train_acc)  # Training accuracy
    results['Test Accuracy'].append(test_acc)  # Testing accuracy

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
print(results)

{'Model': ['Logistic Regression (Word2Vec)', 'Random Forest (Word2Vec)'], 'Train Accuracy': [0.6833824975417896, 0.9783677482792527], 'Test Accuracy': [0.6430155210643016, 0.6629711751662971]}


### 2) FastText

In [ ]:
#Import FastText from Gensim library
from gensim.models import FastText

# Step 1: Train FastText model
# FastText model is trained on tokenized training sentences
# vector_size: Dimensionality of the word vectors
# window: The maximum distance between the current and predicted word within a sentence
# min_count: Ignores all words with total frequency lower than this value
# sg: Training algorithm. 1 means skip-gram, 0 means CBOW
fasttext_model = FastText(
    sentences=X_train_tokenized,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1
)

# Step 2: Function to generate sentence vectors from Word2Vec or FastText models
# This function takes tokenized sentences and converts them into sentence vectors
# by averaging the word vectors for words that exist in the model's vocabulary.
def get_w2v_vectors(data, model, vector_size=100):
    vectors = []
    for sentence in data:
        # Initialize a zero vector for each sentence
        sentence_vec = np.zeros(vector_size)
        count = 0  # To track how many words in the sentence exist in the model
        for word in sentence:
            # Check if the word exists in the model's vocabulary
            if word in model.wv.key_to_index:
                # Add the word vector to the sentence vector
                sentence_vec += model.wv[word]
                count += 1
        # If there are valid words in the sentence, compute the average word vector
        if count != 0:
            # The sentence vector is the average of the word vectors
            sentence_vec /= count
        # Append the sentence vector to the list
        vectors.append(sentence_vec)
    # Return the list of sentence vectors as a numpy array
    return np.array(vectors)

# Step 3: Generate FastText vectors for training and testing data
# Use the previously defined function to convert tokenized sentences to vectors
# by averaging the word vectors learned by the FastText model.
X_train_fasttext = get_w2v_vectors(X_train_tokenized, fasttext_model)
X_test_fasttext = get_w2v_vectors(X_test_tokenized, fasttext_model)

# Step 4: Train machine learning models (Logistic Regression and Random Forest)
# on FastText sentence vectors and evaluate their performance
for model_name, model in models.items():
    # Train the model on FastText vectors and the corresponding labels
    model.fit(X_train_fasttext, y_train)
    # Calculate accuracy on the training and testing data
    train_acc = model.score(X_train_fasttext, y_train)
    test_acc = model.score(X_test_fasttext, y_test)

    # Store the model name and its accuracy results
    results['Model'].append(model_name + " (FastText)")  # Append model name with FastText notation
    results['Train Accuracy'].append(train_acc)  # Append training accuracy
    results['Test Accuracy'].append(test_acc)  # Append testing accuracy


In [ ]:
print(results)

{'Model': ['Logistic Regression (Word2Vec)', 'Random Forest (Word2Vec)', 'Logistic Regression (FastText)', 'Random Forest (FastText)'], 'Train Accuracy': [0.6833824975417896, 0.9783677482792527, 0.7364798426745329, 0.9783677482792527], 'Test Accuracy': [0.6430155210643016, 0.6629711751662971, 0.6866223207686623, 0.696969696969697]}


### 3) GloVe

In [ ]:
# Install mittens library
!pip install mittens

# Import necessary libraries
import numpy as np
from mittens import GloVe
from collections import defaultdict, Counter
from scipy.sparse import dok_matrix
import gc

In [ ]:
# =============================================================================
# Step 1: Build co-occurrence matrix from training data
# =============================================================================

def build_cooccur_matrix(sentences, window_size=5, min_count=5, max_vocab=10000):
    """
    토큰화된 문장들로부터 co-occurrence matrix를 생성합니다.

    Parameters:
    - sentences: 토큰화된 문장들의 리스트
    - window_size: 문맥 윈도우 크기
    - min_count: vocabulary에 포함될 최소 단어 빈도
    - max_vocab: 최대 vocabulary 크기
    """
    # 단어 빈도 계산
    word_counter = Counter()
    for sentence in sentences:
        word_counter.update(sentence)

    # vocabulary 구성 (빈도 기준)
    top_words = [word for word, count in word_counter.most_common(max_vocab)
                 if count >= min_count]
    vocab = set(top_words)
    word2id = {word: idx for idx, word in enumerate(sorted(vocab))}

    print(f'Vocabulary size: {len(vocab)}')

    # Co-occurrence matrix 생성
    cooccur = defaultdict(float)

    for sent_idx, sentence in enumerate(sentences):
        # vocabulary에 있는 단어만 사용
        sentence = [w for w in sentence if w in vocab]

        for i, word in enumerate(sentence):
            start = max(0, i - window_size)
            end = min(len(sentence), i + window_size + 1)

            for j in range(start, end):
                if i != j:
                    context_word = sentence[j]
                    distance = abs(i - j)
                    # 거리에 반비례하는 가중치 적용
                    cooccur[(word2id[word], word2id[context_word])] += 1.0 / distance

        if (sent_idx + 1) % 1000 == 0:
            print(f'Processed {sent_idx + 1}/{len(sentences)} sentences')

    return cooccur, word2id


# Build co-occurrence matrix from training data
print("Building co-occurrence matrix from training data...")
print("=" * 50)

cooccur_dict, word2id = build_cooccur_matrix(
    X_train_tokenized,
    window_size=5,
    min_count=5,
    max_vocab=10000
)

print(f'Co-occurrence pairs: {len(cooccur_dict)}')

Building co-occurrence matrix from training data...
Vocabulary size: 5156
Processed 1000/2034 sentences
Processed 2000/2034 sentences
Co-occurrence pairs: 1024384


In [ ]:
# =============================================================================
# Step 2: Convert to dense matrix and train GloVe model
# =============================================================================

# Convert to dense matrix
vocab_size = len(word2id)
cooccur_dense = np.zeros((vocab_size, vocab_size), dtype=np.float32)

print("\nBuilding dense co-occurrence matrix...")
for (i, j), count in cooccur_dict.items():
    cooccur_dense[i, j] = count

del cooccur_dict
gc.collect()

print(f'Dense matrix shape: {cooccur_dense.shape}')

# Train GloVe model
print("\nTraining GloVe model...")
print("=" * 50)

glove_model = GloVe(
    n=100,              # Vector dimension (no_components)
    max_iter=10,        # epochs
    learning_rate=0.05,
    display_progress=1
)

embeddings = glove_model.fit(cooccur_dense)

print("=" * 50)
print(f'Training completed!')
print(f'Embeddings shape: {embeddings.shape}')

# Create id2word mapping
id2word = {idx: word for word, idx in word2id.items()}


Building dense co-occurrence matrix...
Dense matrix shape: (5156, 5156)

Training GloVe model...


Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Iteration 11: loss: 8024.64404296875

Training completed!
Embeddings shape: (5156, 100)


In [ ]:
# =============================================================================
# Step 3: Function to generate sentence vectors from GloVe embeddings
# =============================================================================

def get_glove_vectors(data, embeddings, word2id, vector_size=100):
    """
    토큰화된 문장들을 GloVe 벡터로 변환합니다.
    각 문장은 단어 벡터들의 평균으로 표현됩니다.

    Parameters:
    - data: 토큰화된 문장들의 리스트
    - embeddings: GloVe word embeddings (numpy array)
    - word2id: 단어에서 인덱스로의 매핑 딕셔너리
    - vector_size: 벡터 차원

    Returns:
    - numpy array of sentence vectors
    """
    vectors = []

    for sentence in data:
        # Initialize a zero vector for each sentence
        sentence_vec = np.zeros(vector_size)
        count = 0  # Count the number of valid words in the vocabulary

        for word in sentence:
            # Check if the word exists in the vocabulary
            if word in word2id:
                # Add the word's GloVe vector to the sentence vector
                word_idx = word2id[word]
                sentence_vec += embeddings[word_idx]
                count += 1

        # If the sentence has valid words, average their word vectors
        if count != 0:
            sentence_vec /= count

        # Append the resulting sentence vector to the list
        vectors.append(sentence_vec)

    return np.array(vectors)


# Generate GloVe vectors for both training and testing data
print("\nGenerating sentence vectors...")
X_train_glove = get_glove_vectors(X_train_tokenized, embeddings, word2id, vector_size=100)
X_test_glove = get_glove_vectors(X_test_tokenized, embeddings, word2id, vector_size=100)

print(f'X_train_glove shape: {X_train_glove.shape}')
print(f'X_test_glove shape: {X_test_glove.shape}')


Generating sentence vectors...
X_train_glove shape: (2034, 100)
X_test_glove shape: (1353, 100)


In [ ]:
# =============================================================================
# Step 4: Model training and performance evaluation
# =============================================================================

# Train each machine learning model on the GloVe sentence vectors
print("\nTraining models with GloVe vectors...")
print("=" * 50)

for model_name, model in models.items():
    print(f'Training {model_name}...')

    # Fit the model using the GloVe vectors and corresponding labels
    model.fit(X_train_glove, y_train)

    # Calculate accuracy on both training and testing datasets
    train_acc = model.score(X_train_glove, y_train)
    test_acc = model.score(X_test_glove, y_test)

    # Store the model name and accuracy results
    results['Model'].append(model_name + " (GloVe)")
    results['Train Accuracy'].append(train_acc)
    results['Test Accuracy'].append(test_acc)

    print(f'  Train Accuracy: {train_acc:.4f}')
    print(f'  Test Accuracy: {test_acc:.4f}')

print("=" * 50)
print("All models trained successfully!")


Training models with GloVe vectors...
Training Logistic Regression...
  Train Accuracy: 0.6981
  Test Accuracy: 0.6726
Training Random Forest...
  Train Accuracy: 0.9784
  Test Accuracy: 0.6526
All models trained successfully!


In [ ]:
# =============================================================================
# Display results
# =============================================================================

import pandas as pd

# Organize the results into a dataframe for better visibility
results_df = pd.DataFrame(results)

# Print the results in a table format
print("\nFinal Results:")
print(results_df)


Final Results:
                            Model  Train Accuracy  Test Accuracy
0  Logistic Regression (Word2Vec)        0.683382       0.643016
1        Random Forest (Word2Vec)        0.978368       0.662971
2  Logistic Regression (FastText)        0.736480       0.686622
3        Random Forest (FastText)        0.978368       0.696970
4     Logistic Regression (GloVe)        0.698132       0.672579
5           Random Forest (GloVe)        0.978368       0.652624
